In [33]:
import pandas as pd
import requests
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [ ]:
raw_df = pd.read_csv('/content/drive/MyDrive/Data_Engineering_Internship/messy_sales_data.csv')
print(f"Rows    : {raw_df.shape[0]}")
print(f"Columns : {raw_df.shape[1]}")
print(f"Column Names : {raw_df.columns.tolist()}")

Rows    : 30
Columns : 9
Column Names : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']


Diagnose: Understanding the problems before fixing
It's necessary step to start before

In [ ]:
#Cecking missing values:
print("\n[1] MISSING VALUES per column:")
print(raw_df.isnull().sum())

#Duplicated rows
print(f"\n[2] DUPLICATED ROWS : {raw_df.duplicated().sum()}")

#Data Types
print("\n[3] DATA TYPES:")
print(raw_df.dtypes)

#Unique values
print("\n[4] UNIQUE CATEGORIES:",raw_df['category'].dropna().unique().tolist())
print("[4] SAMPLE ORDER DATES:", raw_df['order_date'].unique()[:8].tolist())
print("[4] SAMPLE NAMES:", raw_df['customer_name'].unique().tolist())


[1] MISSING VALUES per column:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATED ROWS : 0

[3] DATA TYPES:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATEGORIES: ['Electronics', 'Accessories']
[4] SAMPLE ORDER DATES: ['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15']
[4] SAMPLE NAMES: ['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'kiran mehta', 'Deepak Singh', nan, 'Ananya Das', 'Vikram Iyer', 'Pooja Gupta', 'SURESH RAO', 'Meera Joshi', 'Arjun Nair', 'Tanvi Mehta', 'Kiran Mehta', 'Rohit Verma', 'Sneha Reddy', 'Gaurav Shukla', 'Nisha Kapoor', 'Ajay Tiwari', '

TRANSFORMING THE DATA - REMOVING DUPLICATE VALES , HANDLING NULL VALUES.      

BEFORE PERFORMING ANY ACTIONS DUPLICATE THE DATA SET.

In [ ]:
df = raw_df.copy()
print(f"Copy has been created {df.shape}")

Copy has been created (30, 9)


In [ ]:
print(f"Total missing values before fix: {raw_df.isnull().sum().sum()}")
#.fillna(value) replaces every NAN values with given value
#inplace=True applies changes directly to df
df["customer_name"].fillna('Unknown Customer', inplace=True)

#replacing null values in quantity colum by the median
df["quantity"].fillna(df["quantity"].median(),inplace=True)

#fixing the category colum
df["category"].fillna('Uncategorized',inplace=True)

#fixing the products missing value
df["product"].fillna('Unknown Product',inplace=True)

print(f"Total missing values after fix: {df.isnull().sum().sum()}")
print(f"Total duplicated vales after fix: {df.duplicated().sum()}")



Total missing values before fix: 7
Total missing values after fix: 0
Total duplicated vales after fix: 0


In [ ]:
#Before Duplication
print(f"Rows before duplication:{len(df)}")
print(f"Duplicates rows found :{df.duplicated().sum()}")

#Removal of duplication
#keep = False to marks ALL copies of duplicate as True
duplicate_mask = df.duplicated(keep=False)
print("\nThe duplicate rows(all copies shown):")
print(df[duplicate_mask][['order_id','customer_name','product','order_date']].to_string(index=False))

df.drop_duplicates(inplace=True)
df.reset_index(drop=True,inplace=True)

print(f"\nRows after duplication:{len(df)}")
print(f"Rows removed:{len(df[duplicate_mask])}")

Rows before duplication:30
Duplicates rows found :0

The duplicate rows(all copies shown):
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

Rows after duplication:30
Rows removed:0


In [ ]:
#parse the mixed formats
print("Sample dates before prasing")
print(df['order_date'].head(10).tolist())

#pd.to_datetime() converts dates into poroper datetime objects
#once converted we can extract the year,month,day,day-of-week etc
#errors='coerce' is important
#if a date string cannot be prased it is replaced with NaT instead of crashing
#NaT means "Not a Time" the datetime equivalent to NaN

df['order_date'] = pd.to_datetime(df['order_date'],dayfirst=False,errors='coerce')
nat_mask = df['order_date'].isnull()

Sample dates before prasing
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15', '2024-01-15']


In [ ]:
#handle reamaining NaT values caused by DD-MM-YYYY format
#we try prasing those specific
net_mask = df['order_date'].isnull()
df.loc[nat_mask,'order_date'] = pd.to_datetime(raw_df.loc[nat_mask,'order_date'],dayfirst=True,errors='coerce')

nat_count = df['order_date'].isnull().sum()
print(f"\nunparseable dates remaing (Nat): {nat_count}")

#extract useful date components
# .dt is
df['year'] = df['order_date'].dt.year                   #2024
df['month'] = df['order_date'].dt.month                 #1, 2, 3,....
df['month_name'] = df['order_date'].dt.strftime('%B')   # 'January','February'.....
df['day_name'] = df['order_date'].dt.strftime('%A')     # 'Monday', 'Tuesday'..........

print("\nSample dates after prasing")
print(df[['order_date','year','month','month_name','day_name']].head(10).to_string(index=False))



unparseable dates remaing (Nat): 0

Sample dates after prasing
order_date  year  month month_name  day_name
2024-01-05  2024      1    January    Friday
2024-01-07  2024      1    January    Sunday
2024-01-08  2024      1    January    Monday
2024-01-10  2024      1    January Wednesday
2024-01-05  2024      1    January    Friday
2024-01-07  2024      1    January    Sunday
2024-01-12  2024      1    January    Friday
2024-01-13  2024      1    January  Saturday
2024-01-15  2024      1    January    Monday
2024-01-15  2024      1    January    Monday


In [ ]:
#Standardizing names
print("Names before standardization:")
print(df['customer_name'].unique().tolist())

#Notice "Amit Vera","KIRAN MERA",ramesh kumar" are inconsistent

df['customer_name'] = df['customer_name'].str.strip().str.title()
print("\nNames after standardization:")
print(df['customer_name'].unique().tolist())

Names before standardization:
['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'kiran mehta', 'Deepak Singh', 'Unknown Customer', 'Ananya Das', 'Vikram Iyer', 'Pooja Gupta', 'SURESH RAO', 'Meera Joshi', 'Arjun Nair', 'Tanvi Mehta', 'Kiran Mehta', 'Rohit Verma', 'Sneha Reddy', 'Gaurav Shukla', 'Nisha Kapoor', 'Ajay Tiwari', 'ANANYA DAS', 'Preeti Saxena', 'Amit Bose', 'Rekha Nair', 'Harish Pillai', 'Sanjay Dubey', 'Kavya Nambiar']

Names after standardization:
['Ramesh Kumar', 'Priya Nair', 'Amit Verma', 'Sunita Patel', 'Kiran Mehta', 'Deepak Singh', 'Unknown Customer', 'Ananya Das', 'Vikram Iyer', 'Pooja Gupta', 'Suresh Rao', 'Meera Joshi', 'Arjun Nair', 'Tanvi Mehta', 'Rohit Verma', 'Sneha Reddy', 'Gaurav Shukla', 'Nisha Kapoor', 'Ajay Tiwari', 'Preeti Saxena', 'Amit Bose', 'Rekha Nair', 'Harish Pillai', 'Sanjay Dubey', 'Kavya Nambiar']


In [ ]:
#The data shows keyboard items labbeled as Electronics but keyboard is also considered as a accessory

#1 building boolean mask - true for rows ehere both condition are true
wrong_mask = (df['product'] == 'Keyboard') & (df['category'] == 'Electronics')

print(f"Rows to fix: {wrong_mask.sum()}")
print("Before fix: ")
print(df[wrong_mask][['order_id','product','category']].to_string(index=False))

#2 use .loc to update only the rows where the mask is True
#df.loc[row_condition,'column name'] = new_value

df.loc[wrong_mask,'category'] = 'Accessories'
print("\nAfter fix: ")
print(df[wrong_mask][['order_id','product','category']].to_string(index=False))
print("\nAll unique categories now: ", sorted(df['category'].unique().tolist()))

Rows to fix: 1
Before fix: 
 order_id  product    category
     1024 Keyboard Electronics

After fix: 
 order_id  product    category
     1024 Keyboard Accessories

All unique categories now:  ['Accessories', 'Electronics', 'Uncategorized']


In [ ]:
print(df['category'])

0       Electronics
1       Electronics
2       Accessories
3       Electronics
4       Electronics
5       Accessories
6       Electronics
7       Accessories
8       Electronics
9       Accessories
10      Electronics
11      Accessories
12      Electronics
13      Electronics
14      Accessories
15      Accessories
16      Accessories
17      Electronics
18      Electronics
19      Accessories
20      Electronics
21      Accessories
22      Electronics
23      Accessories
24      Accessories
25      Electronics
26    Uncategorized
27      Electronics
28      Accessories
29      Accessories
Name: category, dtype: object


In [ ]:
#Enforce data types
#pd.to_numeric() converts column into a number type
#errors ='coerce' replaces non numeric values to NaN
#.astype(int) then converts float to integer (removes the decimal,0)

df['quantity'] = pd.to_numeric(df['quantity'], errors = 'coerce').astype('Int64')
df['unit_price'] = pd.to_numeric(df['unit_price'], errors = 'coerce')
df['revenue'] = df['quantity'] * df['unit_price']
print(df[['customer_name','product','quantity','unit_price','revenue']].head(8).to_string(index=False))

   customer_name         product  quantity  unit_price  revenue
    Ramesh Kumar          Laptop         2       45000    90000
      Priya Nair Unknown Product         1       15000    15000
      Amit Verma        Keyboard         3        1200     3600
    Sunita Patel         Monitor         2       22000    44000
    Ramesh Kumar          Laptop         2       45000    90000
     Kiran Mehta           Mouse        10         800     8000
    Deepak Singh      Headphones         2        3500     7000
Unknown Customer          Webcam         1        2500     2500


In [ ]:
# ============================================================
# CELL 11 — Post-Cleaning Validation Report
# ============================================================


# Calculate the data quality score
# We check 5 things: no missing values, no duplicates, no date nulls, no revenue nulls
# Each passing check contributes 20 points (5 checks × 20 = 100)
missing_count   = df.isnull().sum().sum()
duplicate_count = df.duplicated().sum()
date_nulls      = df['order_date'].isnull().sum()
revenue_nulls   = df['revenue'].isnull().sum()


checks_passed   = sum([
    missing_count   == 0,   # 20 points
    duplicate_count == 0,   # 20 points
    date_nulls      == 0,   # 20 points
    revenue_nulls   == 0,   # 20 points
    len(df)         > 0     # 20 points (dataset is not empty)
])
quality_score = checks_passed * 20


# ── Print the report ─────────────────────────────────────────
print("=" * 55)
print("  POST-CLEANING VALIDATION REPORT")
print("=" * 55)
print(f"  Original rows   : {len(raw_df)}")
print(f"  Cleaned rows    : {len(df)}")
print(f"  Rows removed    : {len(raw_df) - len(df)} (duplicates)")
print(f"  Missing values  : {missing_count}")
print(f"  Duplicates      : {duplicate_count}")
print(f"  Date nulls      : {date_nulls}")
print(f"  Revenue nulls   : {revenue_nulls}")
print(f"  Columns total   : {len(df.columns)}")
print("=" * 55)
print(f"  DATA QUALITY SCORE : {quality_score}/100")
print(f"  DATA IS CLEAN      : {quality_score == 100}")
print("=" * 55)


# ── Actionable Debugging Suggestions ─────────────────────────
if missing_count > 0:
    print("\n  ACTION REQUIRED: Missing values detected.")
    print("  → Use df['column'].fillna(value, inplace=True)")
    print("  → For numbers: fillna(df['column'].median())")
    print("  → For text   : fillna('Unknown')")


if duplicate_count > 0:
    print("\n  ACTION REQUIRED: Duplicate rows detected.")
    print("  → Use df.drop_duplicates(inplace=True)")


if date_nulls > 0:
    print("\n  ACTION REQUIRED: Unparseable dates found.")
    print("  → Check for unusual date formats in the raw data")
    print("  → Use pd.to_datetime(col, dayfirst=True, errors='coerce')")


if quality_score == 100:
    print("\n  All checks passed. Data is ready for analysis.")



  POST-CLEANING VALIDATION REPORT
  Original rows   : 30
  Cleaned rows    : 30
  Rows removed    : 0 (duplicates)
  Missing values  : 0
  Duplicates      : 0
  Date nulls      : 0
  Revenue nulls   : 0
  Columns total   : 14
  DATA QUALITY SCORE : 100/100
  DATA IS CLEAN      : True

  All checks passed. Data is ready for analysis.


In [ ]:
output_filename = 'clean_data.csv'
df.to_csv(output_filename,index=False)
print(f"Cleaned data saved to {output_filename}")
print(f"final dataset's shape {df.shape}")

Cleaned data saved to clean_data.csv
final dataset's shape (30, 14)


In [32]:
#SERRPAPI
SERP_API_KEY = '464ddd6c34f6425c8ad48ee1cbbebed05ee82f453a5f7075de7e376e2ece1239'
SERP_URL = 'https://serpapi.com/search.json'
SEARCH_QUERY = 'Data Engineer India'
print(f"SerpAPI Key  : {'Set (live data)' if SERP_API_KEY != 'YOUR_SERPAPI_KEY_HERE' else 'Not set (fallback data will be used)'}")
print(f"Search query: {SEARCH_QUERY}")

SerpAPI Key  : Set (live data)
Search query: Data Engineer India


In [37]:
# ============================================================
# CELL 22 — EXTRACT: Fetch Job Listings from SerpAPI
# ============================================================


def fetch_jobs(query, api_key, num_pages=2):
    """
    Fetches job listings from Google Jobs via SerpAPI.


    Parameters:
        query    (str) : The job search query (e.g. 'Data Engineer India')
        api_key  (str) : Your SerpAPI key
        num_pages(int) : Number of result pages to fetch (default: 2)


    Returns:
        list : A list of job dictionaries
    """
    all_jobs = []


    for page in range(num_pages):
        # API pagination: 'start' tells the API which result to start from
        # Page 0: results 0-9, Page 1: results 10-19, etc.
        params = {
            'engine'    : 'google_jobs',  # Use the Google Jobs search engine
            'q'         : query,
            'api_key'   : api_key,
            'hl'        : 'en',           # Language: English
            'start'     : page * 10       # Pagination offset
        }


        try:
            response = requests.get(SERP_URL, params=params, timeout=15)


            if response.status_code == 200:
                data = response.json()


                # 'jobs_results' is the key in the JSON that holds the job listings
                jobs = data.get('jobs_results', [])


                for job in jobs:
                    # Extract and normalize each job's fields
                    # .get('key', 'default') returns the value if the key exists,
                    # or 'default' if it does not — prevents KeyError crashes
                    all_jobs.append({
                        'title'      : job.get('title', 'Unknown Title'),
                        'company'    : job.get('company_name', 'Unknown Company'),
                        'location'   : job.get('location', 'Unknown Location'),
                        'posted'     : job.get('detected_extensions', {}).get('posted_at', 'Unknown'),
                        'salary'     : job.get('detected_extensions', {}).get('salary', 'Not Disclosed'),
                        'job_type'   : job.get('detected_extensions', {}).get('schedule_type', 'Not Specified'),
                        'description': job.get('description', '')[:300]  # First 300 characters only
                    })


                print(f"  Page {page + 1}: fetched {len(jobs)} jobs")
            else:
                print(f"  Page {page + 1}: API error {response.status_code}")


        except Exception as e:
            print(f"  Page {page + 1}: error — {e}")


    return all_jobs


# ── Actually call the function ────────────────────────────────
job_records = []


if SERP_API_KEY != 'YOUR_SERPAPI_KEY_HERE':
    print(f"Fetching job listings for: '{SEARCH_QUERY}'")
    job_records = fetch_jobs(SEARCH_QUERY, SERP_API_KEY)
    print(f"Total jobs fetched: {len(job_records)}")
else:
    print("No SerpAPI key provided — fallback job data will be loaded next.")





Fetching job listings for: 'Data Engineer India'
  Page 1: fetched 10 jobs
  Page 2: fetched 10 jobs
Total jobs fetched: 20


In [39]:
job_df = pd.DataFrame(job_records)
job_df.to_excel('job_listings.xlsx', index=False)